# RAIDS-NIDS v0.19 external dataset and guard comparison

Run this notebook from the `raids-nids` project root. The protocol is already frozen. Do not change episode families, guard candidates, window boundaries, or model seeds after viewing real outcomes.

In [1]:
from pathlib import Path
import json
import itertools

import raids_nids
import river

from raids_nids.audit import audit_dataset
from raids_nids.config import deep_merge, load_yaml
from raids_nids.guard_benchmark import aggregate_guard_benchmarks, run_guard_benchmark
from raids_nids.unsw_events import build_unsw_event_suite, build_unsw_temporal_cache

print('raids-nids:', raids_nids.__version__)
print('river:', river.__version__)
assert raids_nids.__version__ == '0.1.9'
assert river.__version__ == '0.25.0'

raids-nids: 0.1.9
river: 0.25.0


In [2]:
RAW = Path('data/raw/NF-UNSW-NB15-v3.csv')
CACHE = Path('data/derived/v019_unsw_temporal.npz')
EVENT_DIR = Path('data/derived/v019_unsw_events')
RESULTS_DIR = Path('results/v019_external_guard_comparison/runs')
AGGREGATE_DIR = Path('results/v019_external_guard_comparison/aggregate')

assert RAW.exists(), f'Place the official dataset at: {RAW.resolve()}'
print('Raw dataset:', RAW.resolve())
print('Size (GiB):', round(RAW.stat().st_size / 1024**3, 3))

Raw dataset: C:\Users\ASUS\raids-nids\data\raw\NF-UNSW-NB15-v3.csv
Size (GiB): 0.538


## 1. Build the stable temporal cache

The expected row count is 2,365,424. Stop if the release does not match.

In [3]:
if CACHE.exists() and CACHE.with_suffix('.json').exists():
    cache_report = json.loads(CACHE.with_suffix('.json').read_text(encoding='utf-8'))
    print('Using existing cache:', CACHE.resolve())
else:
    cache_report = build_unsw_temporal_cache(RAW, CACHE)

print(json.dumps(cache_report, indent=2))
assert cache_report['rows'] == 2_365_424

Using existing cache: C:\Users\ASUS\raids-nids\data\derived\v019_unsw_temporal.npz
{
  "cache_sha256": "215b2ea90aa5183c3cd99a20ba5d24c25d1dbe35ebe0f1775ab2889b245f240a",
  "dataset": "NF-UNSW-NB15-v3",
  "family_counts": {
    "Analysis": 1226,
    "Backdoor": 4659,
    "Benign": 2237731,
    "DoS": 5980,
    "Exploits": 42748,
    "Fuzzers": 33816,
    "Generic": 19651,
    "Reconnaissance": 17074,
    "Shellcode": 2381,
    "Worms": 158
  },
  "first_timestamp": "2015-01-22 11:49:36.907000",
  "label_column": "Attack",
  "last_timestamp": "2015-02-18 12:29:24.927000",
  "ordering": "FLOW_START_MILLISECONDS then original zero-based row index",
  "output_cache": "data\\derived\\v019_unsw_temporal.npz",
  "rows": 2365424,
  "source_csv": "data\\raw\\NF-UNSW-NB15-v3.csv",
  "source_sha256": "4ebb97bd74412d566137d95a6fc3ffd8f374f1cf8cfe204d007848e7a668f9b5",
  "time_order_violations_in_raw_order": 335443,
  "timestamp_column": "FLOW_START_MILLISECONDS"
}


## 2. Build all three prespecified episodes

A failed episode remains failed. Do not replace it with another family.

In [4]:
suite = build_unsw_event_suite(
    RAW,
    CACHE,
    EVENT_DIR,
    families=['DoS', 'Exploits', 'Reconnaissance'],
)
print(json.dumps(suite, indent=2))

{
  "protocol_id": "RAIDS-NIDS-v0.19-external-guard-comparison",
  "dataset": "NF-UNSW-NB15-v3",
  "prespecified_families": [
    "DoS",
    "Exploits",
    "Reconnaissance"
  ],
  "replacement_after_outcome": "prohibited",
  "raw_dataset_sha256": "4ebb97bd74412d566137d95a6fc3ffd8f374f1cf8cfe204d007848e7a668f9b5",
  "outcomes": [
    {
      "family": "DoS",
      "status": "failed_event_construction",
      "error_type": "ValueError",
      "reason": "No eligible DoS occurrence satisfied the frozen warm-up and continuity rules; rejected={'insufficient_preceding_rows': 57, 'non_benign_warmup': 5668, 'insufficient_post_rows': 255, 'warmup_gap_limit': 0}"
    },
    {
      "family": "Exploits",
      "status": "failed_event_construction",
      "error_type": "ValueError",
      "reason": "No eligible Exploits occurrence satisfied the frozen warm-up and continuity rules; rejected={'insufficient_preceding_rows': 553, 'non_benign_warmup': 39417, 'insufficient_post_rows': 2778, 'warmup_gap_

## 3. Audit every constructed event

In [5]:
dataset_configs = {
    'DoS': (
        'configs/datasets/nf_unsw_nb15_v3_dos_source.yaml',
        'configs/datasets/nf_unsw_nb15_v3_dos_target.yaml',
    ),
    'Exploits': (
        'configs/datasets/nf_unsw_nb15_v3_exploits_source.yaml',
        'configs/datasets/nf_unsw_nb15_v3_exploits_target.yaml',
    ),
    'Reconnaissance': (
        'configs/datasets/nf_unsw_nb15_v3_reconnaissance_source.yaml',
        'configs/datasets/nf_unsw_nb15_v3_reconnaissance_target.yaml',
    ),
}
constructed = {
    row['family'] for row in suite['outcomes'] if row['status'] == 'constructed'
}
audit_reports = {}
for family in sorted(constructed):
    source_cfg, target_cfg = dataset_configs[family]
    source_report = audit_dataset(source_cfg, Path('results/audits') / f'v019_{family}_source.json')
    target_report = audit_dataset(target_cfg, Path('results/audits') / f'v019_{family}_target.json')
    audit_reports[family] = {'source': source_report, 'target': target_report}
    print(family, 'source rows=', source_report['rows_audited'], 'target rows=', target_report['rows_audited'])

## 4. Run authoritative seed 11

Review each saved score trace and candidate audit before running the remaining seeds. Do not change candidate values.

In [6]:
benchmark_configs = {
    'DoS': 'configs/guard_benchmarks/v019_unsw_dos.yaml',
    'Exploits': 'configs/guard_benchmarks/v019_unsw_exploits.yaml',
    'Reconnaissance': 'configs/guard_benchmarks/v019_unsw_reconnaissance.yaml',
}
seed11_summaries = {}
for family in ['DoS', 'Exploits', 'Reconnaissance']:
    if family not in constructed:
        print(family, 'skipped because event construction failed')
        continue
    summary = run_guard_benchmark(benchmark_configs[family])
    seed11_summaries[family] = summary
    print('\n', family)
    for row in summary['guard_results']:
        print(row['detector'], row['guard_status'], row['post_change_detected'], row['detection_delay_windows'])

DoS skipped because event construction failed
Exploits skipped because event construction failed
Reconnaissance skipped because event construction failed


In [7]:
import numpy as np
import pandas as pd

WARMUP = 20_000
POST = 100_000
MAX_GAP_HOURS = 24.0

with np.load(CACHE, allow_pickle=False) as cache:
    ts = cache["sorted_timestamps"].astype(np.int64)
    codes = cache["sorted_family_codes"].astype(np.int16)
    labels = [str(x) for x in cache["family_labels"].tolist()]

# Longest completely benign interval
benign_mask = codes == labels.index("Benign")
padded = np.concatenate(([False], benign_mask, [False]))
transitions = np.flatnonzero(padded[1:] != padded[:-1])
run_starts = transitions[0::2]
run_stops = transitions[1::2]
run_lengths = run_stops - run_starts

longest = int(np.argmax(run_lengths))
print("Longest all-Benign run:", int(run_lengths[longest]), "flows")
print(
    "Run interval:",
    pd.to_datetime(ts[run_starts[longest]], unit="ms"),
    "to",
    pd.to_datetime(ts[run_stops[longest] - 1], unit="ms"),
)

records = []

for family in ["DoS", "Exploits", "Reconnaissance"]:
    family_code = labels.index(family)
    family_mask = codes == family_code
    positions = np.flatnonzero(family_mask)

    bounded = positions[
        (positions >= WARMUP)
        & (positions + POST <= len(codes))
    ]

    prefix = np.concatenate(
        ([0], np.cumsum(family_mask, dtype=np.int64))
    )

    # The preceding 20,000 flows may contain known attacks,
    # but cannot contain the held-out family.
    heldout_free = bounded[
        (prefix[bounded] - prefix[bounded - WARMUP]) == 0
    ]

    candidates = []

    for raw_position in heldout_free:
        position = int(raw_position)
        start = position - WARMUP

        maximum_gap = float(
            np.diff(ts[start:position + 1]).max() / 3_600_000
        )
        if maximum_gap > MAX_GAP_HOURS:
            continue

        # Every class in the warm-up must already occur in
        # the strictly earlier historical source.
        source_stop = int(
            np.searchsorted(ts, ts[start], side="left")
        )
        historical_counts = np.bincount(
            codes[:source_stop],
            minlength=len(labels),
        )
        warmup_codes = np.unique(codes[start:position])
        minimum_history = int(
            historical_counts[warmup_codes].min()
        )

        if minimum_history == 0:
            continue

        candidates.append({
            "position": position,
            "time": str(pd.to_datetime(ts[position], unit="ms")),
            "warmup_labels": "|".join(
                labels[int(code)] for code in warmup_codes
            ),
            "minimum_history": minimum_history,
            "maximum_gap_hours": round(maximum_gap, 6),
            "post500": int(
                prefix[position + 500] - prefix[position]
            ),
            "post5000": int(
                prefix[position + 5000] - prefix[position]
            ),
            "post100000": int(
                prefix[position + POST] - prefix[position]
            ),
        })

    # Construction-only check for at least 1% held-out-family
    # prevalence in both the first 500 and first 5,000 flows.
    sustained = [
        row for row in candidates
        if row["post500"] >= 5
        and row["post5000"] >= 50
    ]

    first = candidates[0] if candidates else {}
    first_sustained = sustained[0] if sustained else {}

    records.append({
        "family": family,
        "occurrences": len(positions),
        "bounded": len(bounded),
        "no_family_in_prior_20k": len(heldout_free),
        "continuous_and_supported": len(candidates),
        "sustained_1pct": len(sustained),
        "first_supported_time": first.get("time"),
        "warmup_labels": first.get("warmup_labels"),
        "minimum_history": first.get("minimum_history"),
        "post500": first.get("post500"),
        "post5000": first.get("post5000"),
        "post100000": first.get("post100000"),
        "first_1pct_time": first_sustained.get("time"),
    })

print(pd.DataFrame(records).to_string(index=False))

Longest all-Benign run: 945383 flows
Run interval: 2015-01-22 13:47:28.920000 to 2015-01-23 00:25:23.759000
        family  occurrences  bounded  no_family_in_prior_20k  continuous_and_supported  sustained_1pct       first_supported_time                           warmup_labels  minimum_history  post500  post5000  post100000            first_1pct_time
           DoS         5980     5668                       2                         1               0 2015-02-18 01:06:33.228000 Benign|Backdoor|Exploits|Reconnaissance              857        3        16         340                       None
      Exploits        42748    39417                       2                         1               1 2015-02-18 01:04:57.146000                         Benign|Backdoor              824        6       139        2957 2015-02-18 01:04:57.146000
Reconnaissance        17074    15531                       2                         1               1 2015-02-18 01:06:32.190000                Benign|Backd

## 5. Run the remaining paired seeds

Set `RUN_FULL_MATRICES = True` only after checking the seed-11 files.

In [ ]:
RUN_FULL_MATRICES = False

matrix_configs = {
    'DoS': 'configs/matrices/v019_unsw_dos_guards.yaml',
    'Exploits': 'configs/matrices/v019_unsw_exploits_guards.yaml',
    'Reconnaissance': 'configs/matrices/v019_unsw_reconnaissance_guards.yaml',
}
if RUN_FULL_MATRICES:
    for family in ['DoS', 'Exploits', 'Reconnaissance']:
        if family not in constructed:
            continue
        matrix = load_yaml(matrix_configs[family])
        base = load_yaml(matrix['base_benchmark'])
        for combination in itertools.product(*matrix['axes'].values()):
            override = {}
            for value in combination:
                override = deep_merge(override, value)
            summary = run_guard_benchmark(deep_merge(base, override))
            print(family, 'seed', summary['seed'], 'completed')
else:
    print('Full matrices are paused. Review seed-11 evidence first.')

## 6. Aggregate after all eligible matrices finish

In [ ]:
if RUN_FULL_MATRICES:
    aggregate_manifest = aggregate_guard_benchmarks(RESULTS_DIR, AGGREGATE_DIR)
    print(json.dumps(aggregate_manifest, indent=2))
else:
    print('Aggregation is paused until the full matrices finish.')